# Explainable AI for Customer Churn Prediction

> **Notebook status:** This notebook is an **exploratory and extended development notebook**.
>
> **For final dissertation submission (ordered evidence flow), use:**
> `notebooks/churn_xai_dissertation_ordered_submission.ipynb`
>
> The ordered submission notebook follows the strict sequence:
> **all-model comparison -> final model selection -> explainability on selected model -> business outputs**.

**Dissertation title:** *Explainable Artificial Intelligence for Customer Churn Prediction: A Data-Driven and Ethical Machine Learning Study*

This notebook provides a complete MSc-level workflow for:
1. Data loading and quality checks
2. Exploratory data analysis (EDA)
3. Data preprocessing and train-test split
4. Predictive modelling (Logistic Regression, Random Forest, SVM)
5. Model evaluation with classification metrics and ROC analysis
6. Explainable AI (feature importance + SHAP)
7. Advanced model extensions (Gradient Boosting family)
8. Dissertation-ready outputs (saved figures/tables)

---

## How to use this notebook
- Place your CSV file in the project root or update `DATA_PATH` below.
- Run cells in order from top to bottom.
- All dissertation outputs are saved in `outputs/figures/` and `outputs/tables/`.


In [ ]:
# =============================
# 1. IMPORTS AND CONFIGURATION
# =============================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
)

# Optional advanced model (XGBoost) if installed
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

import shap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Paths
BASE_DIR = Path(".").resolve()
DATA_PATH = BASE_DIR / "churn.csv"  # Change this if your file has a different name/path
FIG_DIR = BASE_DIR / "outputs" / "figures"
TABLE_DIR = BASE_DIR / "outputs" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("Base directory:", BASE_DIR)
print("Data path:", DATA_PATH)
print("Figures will be saved to:", FIG_DIR)
print("Tables will be saved to:", TABLE_DIR)
print("XGBoost available:", XGBOOST_AVAILABLE)


## Section A: Data Loading

### What this section does
- Reads the synthetic churn dataset from a CSV file.
- Performs basic checks (shape, sample rows, column names, and data types).

### Why this is important
- Establishes a reliable starting point for analysis.
- Identifies issues early (missing values, incorrect data types, naming inconsistencies).

### Dissertation writing guidance
- **Methodology:** Describe data source, synthetic/anonymised nature, and variable overview.
- **Results:** Briefly report dataset size and key structural observations.


In [ ]:
# ==================
# 2. LOAD THE DATA
# ==================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Please place your CSV there or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

df.head()


## Section B: Exploratory Data Analysis (EDA)

### What this section does
- Summarises descriptive statistics and missing values.
- Visualises target class balance (churn vs non-churn).
- Explores distributions and key relationships.

### Why this is important
- Helps understand data behaviour before modelling.
- Reveals potential imbalance and informative patterns.

### Dissertation writing guidance
- **Methodology:** Explain EDA steps and rationale for plotting choices.
- **Results:** Present main patterns with references to figures.


In [ ]:
# =====================================
# 3. BASIC QUALITY CHECKS + EDA TABLES
# =====================================

# Standardise column names for easier handling
# (keeps meaning intact but avoids spaces/odd characters issues)
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# Candidate target names (supports common churn datasets)
possible_targets = ["Exited", "Churn", "churn", "Target", "target"]
target_col = next((c for c in possible_targets if c in df.columns), None)

if target_col is None:
    raise ValueError(
        "Target column not found. Expected one of: "
        f"{possible_targets}. Found columns: {df.columns.tolist()}"
    )

print("Using target column:", target_col)

# Missing values summary
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary = missing_summary.to_frame(name="missing_count")
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df) * 100).round(2)

missing_summary.to_csv(TABLE_DIR / "table_missing_values.csv")

print("Saved missing value summary to table_missing_values.csv")
missing_summary.head(15)


In [ ]:
# =================================
# 4. EDA VISUALISATIONS (SAVE PLOTS)
# =================================

# 4.1 Target distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(x=target_col, data=df, palette="Set2")
ax.set_title("Target Class Distribution")
ax.set_xlabel("Churn (1 = Yes, 0 = No)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_target_distribution.png", dpi=300)
plt.show()

# 4.2 Numeric feature distributions
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

# Keep core numeric columns often present in churn datasets (if available)
preferred_numeric = [
    "CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"
]
plot_numeric = [c for c in preferred_numeric if c in numeric_cols]

if len(plot_numeric) == 0:
    plot_numeric = numeric_cols[:6]  # fallback

fig, axes = plt.subplots(len(plot_numeric), 1, figsize=(8, 3 * len(plot_numeric)))
if len(plot_numeric) == 1:
    axes = [axes]

for ax, col in zip(axes, plot_numeric):
    sns.histplot(df[col], kde=True, ax=ax, color="#4C72B0")
    ax.set_title(f"Distribution of {col}")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_numeric_distributions.png", dpi=300)
plt.show()

# 4.3 Correlation heatmap (numeric only)
if len(numeric_cols) > 1:
    plt.figure(figsize=(10, 8))
    corr = df[numeric_cols + [target_col]].corr(numeric_only=True)
    sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap (Numeric Features + Target)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_correlation_heatmap.png", dpi=300)
    plt.show()


## Section C: Data Preprocessing and Train-Test Split

### What this section does
- Separates features (`X`) and target (`y`).
- Splits data into training and test sets.
- Builds preprocessing pipelines:
  - Numeric: imputation + standardisation
  - Categorical: imputation + one-hot encoding

### Why this is important
- Prevents data leakage by fitting preprocessing only on training data.
- Ensures fair and reproducible model evaluation.

### Dissertation writing guidance
- **Methodology:** Explain split ratio, stratification, and preprocessing choices.
- **Results:** Mention that identical preprocessing was used for model comparability.


In [ ]:
# ========================================
# 5. PREPROCESSING + TRAIN/TEST SPLIT
# ========================================

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# Remove row identifiers if present
id_like_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in X.columns]
if id_like_cols:
    X = X.drop(columns=id_like_cols)
    print("Dropped non-predictive identifier columns:", id_like_cols)

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)


## Section D: Baseline Model Training

### What this section does
- Trains three widely used classifiers:
  1. Logistic Regression
  2. Random Forest
  3. Support Vector Machine (SVM)

### Why this is important
- Combines interpretable linear modelling (Logistic Regression) with non-linear methods.
- Supports robust comparative analysis.

### Dissertation writing guidance
- **Methodology:** Justify model selection and hyperparameter defaults.
- **Results:** Report comparative metric performance and identify best model.


In [ ]:
# ==========================
# 6. MODEL TRAINING
# ==========================

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
}

trained_pipelines = {}

for name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    clf.fit(X_train, y_train)
    trained_pipelines[name] = clf
    print(f"Trained: {name}")


## Section E: Model Evaluation

### What this section does
- Calculates core classification metrics: accuracy, precision, recall, F1-score, ROC-AUC.
- Generates confusion matrices and ROC curves for model comparison.
- Saves results as tables and publication-quality figures.

### Why this is important
- Single metrics can be misleading in churn prediction (especially with class imbalance).
- Multiple metrics provide a balanced view of performance.

### Dissertation writing guidance
- **Methodology:** Define each metric and explain threshold-based classification.
- **Results:** Compare models using both numeric tables and visual evidence.


In [ ]:
# ==========================
# 7. MODEL EVALUATION
# ==========================

def evaluate_model(name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
    }

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    return metrics, cm, report, fpr, tpr

all_metrics = []
roc_data = {}
conf_matrices = {}

for model_name, model_pipeline in trained_pipelines.items():
    metrics, cm, report, fpr, tpr = evaluate_model(model_name, model_pipeline, X_test, y_test)
    all_metrics.append(metrics)
    roc_data[model_name] = (fpr, tpr, metrics["ROC_AUC"])
    conf_matrices[model_name] = cm

    print("\n" + "="*60)
    print(f"Classification report: {model_name}")
    print(report)

metrics_df = pd.DataFrame(all_metrics).sort_values(by="F1", ascending=False)
metrics_df.to_csv(TABLE_DIR / "table_model_metrics_baseline.csv", index=False)
metrics_df


In [ ]:
# ================================================
# 8. CONFUSION MATRICES + ROC CURVES (SAVE FIGURES)
# ================================================

# Confusion matrices
n_models = len(conf_matrices)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (model_name, cm) in zip(axes, conf_matrices.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(f"Confusion Matrix\n{model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_confusion_matrices_baseline.png", dpi=300)
plt.show()

# ROC curves
plt.figure(figsize=(8, 6))
for model_name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves: Baseline Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_roc_curves_baseline.png", dpi=300)
plt.show()


## Section F: Explainable AI (Feature Importance + SHAP)

### What this section does
- Uses model-based feature importance (Random Forest).
- Uses SHAP values for global feature impact and local explanations.

### Why this is important
- Improves transparency and interpretability of churn predictions.
- Supports ethical and accountable use of AI in business settings.

### Dissertation writing guidance
- **Methodology:** Describe SHAP conceptually (Shapley values from game theory).
- **Results:** Report top drivers of churn and discuss business/ethical implications.


In [ ]:
# =================================
# 9. FEATURE IMPORTANCE (RANDOM FOREST)
# =================================

rf_pipe = trained_pipelines["Random Forest"]
rf_model = rf_pipe.named_steps["model"]
rf_pre = rf_pipe.named_steps["preprocessor"]

# Get feature names after preprocessing
feature_names = rf_pre.get_feature_names_out()
rf_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

rf_importances.to_csv(TABLE_DIR / "table_random_forest_feature_importance.csv", index=False)

top_n = 20
plt.figure(figsize=(10, 7))
sns.barplot(
    data=rf_importances.head(top_n),
    x="importance",
    y="feature",
    palette="viridis"
)
plt.title(f"Top {top_n} Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_random_forest_feature_importance_top20.png", dpi=300)
plt.show()

rf_importances.head(10)


In [ ]:
# =========================
# 10. SHAP EXPLAINABILITY
# =========================

# SHAP with tree-based model is computationally efficient using TreeExplainer.
# To keep runtime manageable, we explain a sample from the test set.

sample_size = min(500, len(X_test))
X_test_sample = X_test.sample(sample_size, random_state=RANDOM_STATE)

# Transform sample using trained preprocessor
X_test_sample_transformed = rf_pre.transform(X_test_sample)

# Build SHAP explainer for Random Forest
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_sample_transformed)

# Handle binary classification output shape across SHAP versions
if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_vals_for_class1 = shap_values[1]
else:
    shap_vals_for_class1 = shap_values

# Summary plot (bar)
plt.figure()
shap.summary_plot(
    shap_vals_for_class1,
    features=X_test_sample_transformed,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_shap_summary_bar_top20.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary plot (beeswarm)
plt.figure()
shap.summary_plot(
    shap_vals_for_class1,
    features=X_test_sample_transformed,
    feature_names=feature_names,
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_shap_summary_beeswarm_top20.png", dpi=300, bbox_inches="tight")
plt.show()


## Section G: Advanced Model Extensions

This section adds 2–3 stronger models often used in tabular classification:
1. Gradient Boosting Classifier
2. XGBoost (if installed)
3. (Optional) HistGradientBoosting as a fast alternative

### Why these models may improve performance
- Boosting methods combine weak learners sequentially to reduce bias.
- They often capture complex non-linear interactions better than linear models.
- They frequently perform strongly on structured business datasets.

### Dissertation writing guidance
- **Methodology:** Position these as advanced comparative benchmarks.
- **Results:** Compare against baseline models and discuss trade-off between performance and interpretability.


In [ ]:
# =====================================
# 11. ADVANCED MODELS IMPLEMENTATION
# =====================================

from sklearn.ensemble import HistGradientBoostingClassifier

advanced_models = {
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

if XGBOOST_AVAILABLE:
    # scale_pos_weight can be tuned when classes are imbalanced
    advanced_models["XGBoost"] = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

advanced_pipelines = {}
advanced_metrics = []

for name, model in advanced_models.items():
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    pipe.fit(X_train, y_train)
    advanced_pipelines[name] = pipe

    m, cm, report, fpr, tpr = evaluate_model(name, pipe, X_test, y_test)
    advanced_metrics.append(m)
    print(f"\n{name} report:\n{report}")

advanced_metrics_df = pd.DataFrame(advanced_metrics).sort_values(by="F1", ascending=False)
advanced_metrics_df.to_csv(TABLE_DIR / "table_model_metrics_advanced.csv", index=False)
advanced_metrics_df


In [ ]:
# =============================================
# 12. COMBINED PERFORMANCE TABLE (ALL MODELS)
# =============================================

combined_metrics = pd.concat([metrics_df, advanced_metrics_df], axis=0, ignore_index=True)
combined_metrics = combined_metrics.sort_values(by="F1", ascending=False)
combined_metrics.to_csv(TABLE_DIR / "table_model_metrics_all.csv", index=False)

combined_metrics


## Section H: Ethical, Transparency, and Governance Reflection

### Suggested discussion points for dissertation
- The dataset is synthetic and anonymised, reducing direct privacy risk.
- Model predictions can still introduce **indirect bias** if feature patterns reflect structural inequalities.
- Explainability methods (e.g., SHAP) support transparency and accountability.
- Churn interventions should be monitored to avoid unfair profiling of customer groups.
- Governance should include:
  - periodic model audits,
  - performance monitoring over time,
  - documentation of model updates,
  - clear communication of model limitations.


## Section I: Dissertation Figure/Table Checklist

- **Figures generated by this notebook** (saved in `outputs/figures/`):
  - `fig_target_distribution.png`
  - `fig_numeric_distributions.png`
  - `fig_correlation_heatmap.png`
  - `fig_confusion_matrices_baseline.png`
  - `fig_roc_curves_baseline.png`
  - `fig_random_forest_feature_importance_top20.png`
  - `fig_shap_summary_bar_top20.png`
  - `fig_shap_summary_beeswarm_top20.png`

- **Tables generated by this notebook** (saved in `outputs/tables/`):
  - `table_missing_values.csv`
  - `table_model_metrics_baseline.csv`
  - `table_model_metrics_advanced.csv`
  - `table_model_metrics_all.csv`
  - `table_random_forest_feature_importance.csv`


## Section-by-Section Dissertation Notes (Quick Reference)

### 1) Data loading
- **What:** Import CSV, inspect dimensions/types.
- **Why:** Ensures reproducibility and data integrity.
- **Write-up placement:** Methodology (Data source and preparation).

### 2) EDA
- **What:** Summary statistics, class balance, distributions, correlations.
- **Why:** Identifies patterns and potential modelling challenges.
- **Write-up placement:** Results (Descriptive analysis).

### 3) Preprocessing
- **What:** Imputation, encoding, scaling in a reproducible pipeline.
- **Why:** Ensures proper treatment of mixed data types and prevents leakage.
- **Write-up placement:** Methodology (Preprocessing pipeline).

### 4) Model training
- **What:** Train Logistic Regression, Random Forest, SVM under common split.
- **Why:** Enables fair comparative benchmarking.
- **Write-up placement:** Methodology (Model development).

### 5) Evaluation
- **What:** Accuracy, precision, recall, F1, confusion matrix, ROC-AUC.
- **Why:** Captures multiple dimensions of performance, especially for churn class.
- **Write-up placement:** Results (Comparative performance).

### 6) Explainability (SHAP)
- **What:** Global feature influence and feature impact distribution.
- **Why:** Makes predictions transparent and supports ethical interpretation.
- **Write-up placement:** Results + Discussion (Explainability and trust).

### 7) Advanced models
- **What:** Add boosting-based models for stronger tabular performance.
- **Why:** Tests whether advanced learners improve predictive quality.
- **Write-up placement:** Methodology extension + Results comparison.


# Extended Dissertation Workflow (In-Depth)

This extension expands the notebook into a deeper MSc-level workflow with more cells, richer visuals, stronger modelling, and more detailed explainability.

## Main analytical goals
1. Predict customer churn accurately.
2. Explain *why* customers are predicted to leave.
3. Produce evidence suitable for a long-form dissertation (15,000-18,000 words).

## What is added in this extension
- Additional exploratory and diagnostic analyses
- Feature engineering focused on churn behaviour
- Advanced models and hyperparameter tuning
- Threshold and business-cost analysis
- Calibration analysis
- Deep XAI: permutation importance, SHAP global and local explanations
- Risk-segment and subgroup analysis for ethical transparency


## Extension A: Extended Data Profiling

### What this section adds
- Data quality audit table for all features
- Outlier scan on numeric features
- Explicit class imbalance assessment

### Dissertation relevance
Use this section in **Methodology (Data Profiling)** and **Results (Descriptive Diagnostics)** to justify subsequent preprocessing and modelling decisions.


In [ ]:
# =====================================
# A1. DETAILED DATA AUDIT TABLE
# =====================================

audit_df = pd.DataFrame({
    "feature": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [df[c].isnull().sum() for c in df.columns],
    "missing_percent": [round(df[c].isnull().mean() * 100, 2) for c in df.columns],
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
})

audit_df = audit_df.sort_values(by=["missing_percent", "n_unique"], ascending=[False, False])
audit_df.to_csv(TABLE_DIR / "table_feature_audit_extended.csv", index=False)

print("Saved: table_feature_audit_extended.csv")
audit_df.head(20)


In [ ]:
# =====================================
# A2. CLASS IMBALANCE DIAGNOSTIC
# =====================================

class_counts = y.value_counts().sort_index()
class_percents = (class_counts / len(y) * 100).round(2)

imbalance_table = pd.DataFrame({
    "class": class_counts.index,
    "count": class_counts.values,
    "percent": class_percents.values,
})
imbalance_table.to_csv(TABLE_DIR / "table_class_distribution_extended.csv", index=False)

print("Class distribution:")
display(imbalance_table)

minority_ratio = class_counts.min() / class_counts.max()
print(f"Minority/Majority ratio: {minority_ratio:.3f}")
if minority_ratio < 0.5:
    print("Notice: Class imbalance detected. Precision/Recall/F1 and PR-AUC are particularly important.")


In [ ]:
# =====================================
# A3. OUTLIER SCAN (IQR METHOD)
# =====================================

def iqr_outlier_rate(series: pd.Series) -> float:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return mask.mean()

outlier_rows = []
for c in numeric_cols:
    outlier_rows.append({
        "feature": c,
        "iqr_outlier_rate": round(iqr_outlier_rate(df[c].dropna()), 4)
    })

outlier_df = pd.DataFrame(outlier_rows).sort_values("iqr_outlier_rate", ascending=False)
outlier_df.to_csv(TABLE_DIR / "table_outlier_rates_iqr.csv", index=False)

print("Saved: table_outlier_rates_iqr.csv")
outlier_df.head(15)


## Extension B: Deeper EDA for Churn Drivers

### What this section adds
- Bivariate churn analysis by key categorical and numeric groupings
- Churn rate profiles by age bands, product count, active status, geography and gender
- Additional visuals that support a richer narrative in dissertation results


In [ ]:
# =====================================
# B1. CHURN RATE BY KEY CATEGORICALS
# =====================================

candidate_cats = [
    c for c in [
        "country", "gender", "credit_card", "active_member", "products_number",
        "Geography", "Gender", "HasCrCard", "IsActiveMember", "NumOfProducts"
    ] if c in df.columns
]

for c in candidate_cats:
    churn_by_c = df.groupby(c, dropna=False)[target_col].mean().sort_values(ascending=False)

    plt.figure(figsize=(8, 4))
    sns.barplot(x=churn_by_c.index.astype(str), y=churn_by_c.values, palette="crest")
    plt.title(f"Churn Rate by {c}")
    plt.xlabel(c)
    plt.ylabel("Average churn rate")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"fig_churn_rate_by_{c}_extended.png", dpi=300)
    plt.show()


In [ ]:
# =====================================
# B2. AGE BAND ANALYSIS
# =====================================

if "Age" in df.columns:
    age_bins = [0, 25, 35, 45, 55, 65, 100]
    age_labels = ["<=25", "26-35", "36-45", "46-55", "56-65", "65+"]

    df_age = df.copy()
    df_age["AgeBand"] = pd.cut(df_age["Age"], bins=age_bins, labels=age_labels, right=True, include_lowest=True)

    ageband_churn = df_age.groupby("AgeBand")[target_col].mean().reset_index()
    ageband_churn.columns = ["AgeBand", "ChurnRate"]
    ageband_churn.to_csv(TABLE_DIR / "table_churn_by_ageband.csv", index=False)

    plt.figure(figsize=(8, 4))
    sns.barplot(data=ageband_churn, x="AgeBand", y="ChurnRate", palette="magma")
    plt.title("Churn Rate by Age Band")
    plt.xlabel("Age band")
    plt.ylabel("Churn rate")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_churn_rate_by_ageband.png", dpi=300)
    plt.show()


In [ ]:
# =====================================
# B3. TWO-WAY HEATMAP: GEOGRAPHY/COUNTRY x ACTIVE MEMBER
# =====================================

geo_col = "Geography" if "Geography" in df.columns else ("country" if "country" in df.columns else None)
active_col = "IsActiveMember" if "IsActiveMember" in df.columns else ("active_member" if "active_member" in df.columns else None)

if geo_col is not None and active_col is not None:
    pivot = df.pivot_table(
        values=target_col,
        index=geo_col,
        columns=active_col,
        aggfunc="mean"
    )
    pivot.to_csv(TABLE_DIR / "table_churn_geography_active_pivot.csv")

    plt.figure(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd")
    plt.title(f"Churn Rate Heatmap: {geo_col} x {active_col}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_churn_heatmap_geography_active.png", dpi=300)
    plt.show()


## Extension C: Advanced Modelling and Robust Validation

### What this section adds
- Additional performance metrics for imbalanced settings (PR-AUC, balanced interpretation)
- Optional hyperparameter tuning for selected advanced models
- Calibration and threshold optimisation analysis

### Dissertation relevance
Use this for **Methodology (Model optimisation and validation)** and **Results (Model diagnostics and business decision thresholds)**.


In [ ]:
# =====================================
# C1. EXTENDED METRIC FUNCTION
# =====================================

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    matthews_corrcoef,
    precision_recall_curve,
)

def evaluate_model_extended(name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "PR_AUC": average_precision_score(y_test, y_proba),
        "Brier": brier_score_loss(y_test, y_proba),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    return metrics, cm, fpr, tpr, precision, recall, y_proba


In [ ]:
# =====================================
# C2. OPTIONAL TUNING: RANDOM FOREST / HISTGB / XGBOOST
# =====================================

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tuned_candidates = {
    "Random Forest (Tuned)": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            "model__n_estimators": [200, 400, 600],
            "model__max_depth": [None, 6, 10, 16],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__class_weight": [None, "balanced"]
        }
    ),
    "HistGradientBoosting (Tuned)": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_iter": [200, 400, 600],
            "model__max_depth": [None, 4, 8],
            "model__min_samples_leaf": [20, 40, 80],
            "model__l2_regularization": [0.0, 0.1, 1.0]
        }
    ),
}

if XGBOOST_AVAILABLE:
    tuned_candidates["XGBoost (Tuned)"] = (
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        {
            "model__n_estimators": [200, 400, 600],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [3, 4, 6],
            "model__subsample": [0.8, 0.9, 1.0],
            "model__colsample_bytree": [0.8, 0.9, 1.0]
        }
    )

tuned_pipelines = {}
tuning_summary = []

for name, (model_obj, param_dist) in tuned_candidates.items():
    print(f"\nTuning: {name}")
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model_obj)
    ])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=15,
        scoring="f1",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0
    )

    search.fit(X_train, y_train)

    tuned_pipelines[name] = search.best_estimator_
    tuning_summary.append({
        "Model": name,
        "BestCVScore_F1": search.best_score_,
        "BestParams": str(search.best_params_)
    })

    print("Best CV F1:", round(search.best_score_, 4))

if len(tuning_summary) > 0:
    tuning_df = pd.DataFrame(tuning_summary).sort_values("BestCVScore_F1", ascending=False)
    tuning_df.to_csv(TABLE_DIR / "table_hyperparameter_tuning_summary.csv", index=False)
    display(tuning_df)


In [ ]:
# =====================================
# C3. MERGE ALL MODELS (BASELINE + ADVANCED + TUNED)
# =====================================

all_candidate_models = {}
all_candidate_models.update(trained_pipelines)
all_candidate_models.update(advanced_pipelines)
all_candidate_models.update(tuned_pipelines)

print("Total candidate models:", len(all_candidate_models))
print("Model list:")
for m in all_candidate_models.keys():
    print(" -", m)


In [ ]:
# =====================================
# C4. EVALUATE ALL CANDIDATES WITH EXTENDED METRICS
# =====================================

all_eval_rows = []
all_roc = {}
all_pr = {}
all_cm = {}
all_proba = {}

for model_name, pipe in all_candidate_models.items():
    m, cm, fpr, tpr, prec, rec, y_proba = evaluate_model_extended(model_name, pipe, X_test, y_test)
    all_eval_rows.append(m)
    all_cm[model_name] = cm
    all_roc[model_name] = (fpr, tpr, m["ROC_AUC"])
    all_pr[model_name] = (rec, prec, m["PR_AUC"])
    all_proba[model_name] = y_proba

all_eval_df = pd.DataFrame(all_eval_rows).sort_values(by=["F1", "ROC_AUC"], ascending=[False, False])
all_eval_df.to_csv(TABLE_DIR / "table_all_models_extended_metrics.csv", index=False)

print("Saved: table_all_models_extended_metrics.csv")
all_eval_df.head(15)


In [ ]:
# =====================================
# C5. VISUAL: ROC + PR CURVES FOR TOP MODELS
# =====================================

top_k = min(6, len(all_eval_df))
top_models = all_eval_df.head(top_k)["Model"].tolist()

# ROC
plt.figure(figsize=(8, 6))
for m in top_models:
    fpr, tpr, auc = all_roc[m]
    plt.plot(fpr, tpr, label=f"{m} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Top Models)")
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_roc_curves_top_models_extended.png", dpi=300)
plt.show()

# PR
plt.figure(figsize=(8, 6))
for m in top_models:
    rec, prec, ap = all_pr[m]
    plt.plot(rec, prec, label=f"{m} (AP={ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves (Top Models)")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_pr_curves_top_models_extended.png", dpi=300)
plt.show()


In [ ]:
# =====================================
# C6. TOP MODEL SELECTION + CALIBRATION DIAGNOSTIC
# =====================================

best_model_name = all_eval_df.iloc[0]["Model"]
best_model_pipe = all_candidate_models[best_model_name]

print("Selected best model:", best_model_name)

from sklearn.calibration import calibration_curve

best_proba = all_proba[best_model_name]
prob_true, prob_pred = calibration_curve(y_test, best_proba, n_bins=10, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker="o", label="Model")
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title(f"Calibration Curve: {best_model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_calibration_curve_best_model.png", dpi=300)
plt.show()


In [ ]:
# =====================================
# C7. THRESHOLD OPTIMISATION + BUSINESS COST ANALYSIS
# =====================================

# Simple cost framework example:
# False Negative (missed churner) cost is often higher than False Positive.
COST_FN = 5
COST_FP = 1

thresholds = np.arange(0.05, 0.96, 0.01)
rows = []

for t in thresholds:
    pred_t = (best_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()

    precision_t = precision_score(y_test, pred_t, zero_division=0)
    recall_t = recall_score(y_test, pred_t, zero_division=0)
    f1_t = f1_score(y_test, pred_t, zero_division=0)
    total_cost = (fn * COST_FN) + (fp * COST_FP)

    rows.append({
        "threshold": t,
        "precision": precision_t,
        "recall": recall_t,
        "f1": f1_t,
        "fp": fp,
        "fn": fn,
        "business_cost": total_cost
    })

threshold_df = pd.DataFrame(rows)
threshold_df.to_csv(TABLE_DIR / "table_threshold_business_cost_analysis.csv", index=False)

best_f1_thr = threshold_df.loc[threshold_df["f1"].idxmax(), "threshold"]
best_cost_thr = threshold_df.loc[threshold_df["business_cost"].idxmin(), "threshold"]

print(f"Best F1 threshold: {best_f1_thr:.2f}")
print(f"Lowest-cost threshold: {best_cost_thr:.2f}")

plt.figure(figsize=(9, 5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1", linewidth=2)
plt.axvline(best_f1_thr, linestyle="--", color="black", label=f"Best F1 thr={best_f1_thr:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Metric")
plt.title(f"Threshold Sensitivity: {best_model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_threshold_sensitivity_best_model.png", dpi=300)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(threshold_df["threshold"], threshold_df["business_cost"], color="#C44E52", linewidth=2)
plt.axvline(best_cost_thr, linestyle="--", color="black", label=f"Min cost thr={best_cost_thr:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Business cost")
plt.title("Threshold vs Business Cost")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_threshold_business_cost_curve.png", dpi=300)
plt.show()


## Extension D: In-Depth Explainable AI (XAI)

### What this section adds
- Model-agnostic permutation importance
- SHAP global explanations (bar and beeswarm)
- SHAP dependence analysis for key features
- Local SHAP case explanations for individual customers

### Dissertation relevance
Use this in **Results (Explainability Findings)** and **Discussion (Ethical transparency and actionable insight)**.


In [ ]:
# =====================================
# D1. PERMUTATION IMPORTANCE (MODEL-AGNOSTIC)
# =====================================

from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    estimator=best_model_pipe,
    X=X_test,
    y=y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scoring="f1"
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False)

perm_df.to_csv(TABLE_DIR / "table_permutation_importance_best_model_extended.csv", index=False)

plt.figure(figsize=(9, 7))
sns.barplot(data=perm_df.head(20), x="importance_mean", y="feature", palette="rocket")
plt.title(f"Top 20 Permutation Importances ({best_model_name})")
plt.xlabel("Mean importance (F1 decrease)")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_permutation_importance_top20_best_model_extended.png", dpi=300)
plt.show()

perm_df.head(20)


In [ ]:
# =====================================
# D2. SHAP ON A TREE-BASED PIPELINE (PREFERRED)
# =====================================

# SHAP TreeExplainer works best with tree models.
# If best model is non-tree, we fallback to Random Forest from trained models.

def is_tree_model_name(name: str) -> bool:
    key = name.lower()
    return any(k in key for k in ["forest", "boost", "xgboost", "hist", "tree"])

if is_tree_model_name(best_model_name):
    shap_model_name = best_model_name
    shap_pipe = best_model_pipe
else:
    shap_model_name = "Random Forest"
    shap_pipe = trained_pipelines.get("Random Forest", best_model_pipe)

print("SHAP model used:", shap_model_name)

shap_pre = shap_pipe.named_steps["preprocessor"]
shap_model = shap_pipe.named_steps["model"]

feature_names_shap = shap_pre.get_feature_names_out()
X_test_trans = shap_pre.transform(X_test)

# Convert sparse to dense if needed
if hasattr(X_test_trans, "toarray"):
    X_test_trans = X_test_trans.toarray()

sample_n = min(400, X_test_trans.shape[0])
sample_idx = np.random.RandomState(RANDOM_STATE).choice(X_test_trans.shape[0], sample_n, replace=False)
X_shap = X_test_trans[sample_idx]

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_matrix = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_matrix = shap_values[:, :, 1]
else:
    shap_matrix = shap_values

print("SHAP matrix shape:", np.array(shap_matrix).shape)


In [ ]:
# =====================================
# D3. SHAP GLOBAL PLOTS (BAR + BEESWARM)
# =====================================

plt.figure()
shap.summary_plot(
    shap_matrix,
    X_shap,
    feature_names=feature_names_shap,
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_shap_global_bar_extended.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure()
shap.summary_plot(
    shap_matrix,
    X_shap,
    feature_names=feature_names_shap,
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_shap_global_beeswarm_extended.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# =====================================
# D4. SHAP DEPENDENCE PLOTS FOR TOP FEATURES
# =====================================

mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
shap_rank_idx = np.argsort(mean_abs_shap)[::-1]
top_shap_features = [feature_names_shap[i] for i in shap_rank_idx[:3]]

print("Top SHAP features:", top_shap_features)

for feat in top_shap_features:
    plt.figure()
    shap.dependence_plot(feat, shap_matrix, X_shap, feature_names=feature_names_shap, show=False)
    plt.tight_layout()
    safe_feat = feat.replace("__", "_").replace("/", "_")
    plt.savefig(FIG_DIR / f"fig_shap_dependence_{safe_feat}.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# =====================================
# D5. LOCAL EXPLANATIONS: TOP RISK CUSTOMERS
# =====================================

# Use best model probabilities to identify high-risk customers
high_risk_df = X_test.copy()
high_risk_df["true_churn"] = y_test.values
high_risk_df["pred_proba"] = best_proba
high_risk_df = high_risk_df.sort_values("pred_proba", ascending=False).reset_index(drop=True)

# Save top 20 risk cases
high_risk_df.head(20).to_csv(TABLE_DIR / "table_top20_high_risk_customers.csv", index=False)

print("Saved: table_top20_high_risk_customers.csv")
high_risk_df.head(10)


In [ ]:
# =====================================
# D6. SHAP WATERFALL FOR SELECTED CASES
# =====================================

# For local SHAP plots, we explain 2 examples from SHAP sample indices.
# Case A: highest predicted probability among sampled points
# Case B: median predicted probability among sampled points

# map sampled rows back to best model probabilities if needed
# here we use shap model output context directly for local explanation

if isinstance(explainer.expected_value, (list, np.ndarray)):
    base_value = np.array(explainer.expected_value).ravel()[-1]
else:
    base_value = explainer.expected_value

# define representative indices in X_shap space
local_idx_a = int(np.argmax(np.abs(shap_matrix).sum(axis=1)))
local_idx_b = int(np.argsort(np.abs(shap_matrix).sum(axis=1))[len(shap_matrix)//2])

for idx_local, tag in [(local_idx_a, "caseA"), (local_idx_b, "caseB")]:
    exp_obj = shap.Explanation(
        values=shap_matrix[idx_local],
        base_values=base_value,
        data=X_shap[idx_local],
        feature_names=feature_names_shap
    )

    plt.figure()
    shap.plots.waterfall(exp_obj, max_display=15, show=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"fig_shap_waterfall_{tag}.png", dpi=300, bbox_inches="tight")
    plt.show()


## Extension E: Risk Segmentation and Actionability

### What this section adds
- Customer risk buckets from predicted probabilities
- Segment profiling to identify patterns among high-risk customers
- Practical retention insights linked to explainability findings

### Dissertation relevance
Use this section for **Discussion (managerial implications)** and **Recommendations**.


In [ ]:
# =====================================
# E1. CREATE RISK SEGMENTS
# =====================================

risk_df = X_test.copy()
risk_df["true_churn"] = y_test.values

# Use probabilities from the selected best model in this section
best_proba_for_segments = all_proba[best_model_name]
risk_df["pred_proba"] = best_proba_for_segments

# Risk bucket strategy (can be adapted)
risk_bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
risk_labels = ["Very Low", "Low", "Medium", "High", "Very High"]
risk_df["risk_segment"] = pd.cut(risk_df["pred_proba"], bins=risk_bins, labels=risk_labels, include_lowest=True)

segment_table = risk_df.groupby("risk_segment").agg(
    n_customers=("pred_proba", "size"),
    avg_pred_proba=("pred_proba", "mean"),
    observed_churn_rate=("true_churn", "mean")
).reset_index()

segment_table.to_csv(TABLE_DIR / "table_risk_segment_summary.csv", index=False)
segment_table


In [ ]:
# =====================================
# E2. VISUALISE RISK SEGMENTS
# =====================================

plt.figure(figsize=(8, 4))
sns.barplot(data=segment_table, x="risk_segment", y="n_customers", palette="Blues")
plt.title("Customer Count by Predicted Risk Segment")
plt.xlabel("Risk segment")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_risk_segment_counts.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=segment_table, x="risk_segment", y="observed_churn_rate", palette="Reds")
plt.title("Observed Churn Rate by Predicted Risk Segment")
plt.xlabel("Risk segment")
plt.ylabel("Observed churn rate")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_risk_segment_observed_churn.png", dpi=300)
plt.show()


In [ ]:
# =====================================
# E3. PROFILE HIGH-RISK CUSTOMERS
# =====================================

high_risk_only = risk_df[risk_df["risk_segment"].isin(["High", "Very High"])].copy()

profile_rows = []
for c in numeric_cols:
    if c in high_risk_only.columns:
        profile_rows.append({
            "feature": c,
            "high_risk_mean": high_risk_only[c].mean(),
            "overall_mean": risk_df[c].mean(),
            "difference": high_risk_only[c].mean() - risk_df[c].mean(),
        })

profile_df = pd.DataFrame(profile_rows).sort_values("difference", key=lambda s: np.abs(s), ascending=False)
profile_df.to_csv(TABLE_DIR / "table_high_risk_profile_numeric.csv", index=False)

print("Saved: table_high_risk_profile_numeric.csv")
profile_df.head(15)


## Extension F: Structured Interpretation Notes (for Writing)

Use these prompts directly in your dissertation text:

1. **Prediction performance**
   - Which model had the strongest F1 and ROC-AUC?
   - Was there a trade-off between precision and recall?

2. **Churn drivers (global explanation)**
   - Which features consistently appeared in top-10 importance/SHAP?
   - Are these drivers behaviourally plausible in banking churn context?

3. **Customer-level explanation (local explanation)**
   - For a high-risk case, which top 3 factors increased churn probability?
   - Are those factors actionable by retention teams?

4. **Ethics and governance**
   - Were subgroup performance differences observed?
   - What governance controls are needed (monitoring, audit, documentation)?


## Final Expanded Notebook Checklist

This expanded notebook now supports a long dissertation by providing:

- Extensive EDA and diagnostic tables
- Baseline, advanced, and tuned models
- Multiple validation views (ROC, PR, calibration, threshold-cost)
- In-depth XAI (permutation + SHAP global + SHAP local)
- Risk segmentation and actionable customer profiling

### Suggested chapter mapping
- **Chapter 3 (Methodology):** preprocessing, model design, tuning, evaluation protocol
- **Chapter 4 (Results):** EDA findings, model performance, explainability figures
- **Chapter 5 (Discussion):** business interpretation, ethics, limitations, recommendations


# Goal-Focused Dissertation Section (Human-Centred Narrative)

This section is written around the 4 core goals of your study:

1. **Prediction** → Which customers are likely to churn?  
2. **Explanation** → Why are they likely to churn?  
3. **Insight** → What factors influence churn?  
4. **Value** → How can businesses use this information?

The aim is to produce outputs that are easy to explain in dissertation chapters and understandable to non-technical stakeholders.


## 1️⃣ Prediction

### Cell purpose
This block identifies customers most likely to churn and quantifies model confidence.

### Why it matters
In churn management, prediction is the first step: firms need a ranked list of at-risk customers before intervention planning.


In [ ]:
# =====================================
# G1. RANK CUSTOMERS BY CHURN PROBABILITY
# =====================================

prediction_df = X_test.copy()
prediction_df["true_churn"] = y_test.values
prediction_df["predicted_probability"] = all_proba[best_model_name]
prediction_df["predicted_label_0_5"] = (prediction_df["predicted_probability"] >= 0.5).astype(int)

# Top likely churners
top_k = 100
top_risk_customers = prediction_df.sort_values("predicted_probability", ascending=False).head(top_k)
top_risk_customers.to_csv(TABLE_DIR / "table_top_100_likely_churners.csv", index=False)

print(f"Top {top_k} likely churners saved to table_top_100_likely_churners.csv")
top_risk_customers.head(10)


In [ ]:
# =====================================
# G2. PROBABILITY DISTRIBUTION: CHURN VS NON-CHURN
# =====================================

plt.figure(figsize=(8, 5))
sns.kdeplot(
    data=prediction_df,
    x="predicted_probability",
    hue="true_churn",
    common_norm=False,
    fill=True,
    alpha=0.35,
)
plt.title(f"Predicted Churn Probability Distribution ({best_model_name})")
plt.xlabel("Predicted churn probability")
plt.ylabel("Density")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_probability_distribution_true_classes.png", dpi=300)
plt.show()


In [ ]:
# =====================================
# G3. DECILE LIFT TABLE (BUSINESS-FRIENDLY PREDICTION VIEW)
# =====================================

# Deciles show how concentrated true churners are in highest-risk bins.
decile_df = prediction_df.copy()
decile_df["decile"] = pd.qcut(decile_df["predicted_probability"], q=10, labels=False, duplicates="drop")
decile_df["decile"] = decile_df["decile"].max() - decile_df["decile"] + 1  # decile 1 = highest risk

decile_table = decile_df.groupby("decile").agg(
    customers=("true_churn", "size"),
    churners=("true_churn", "sum"),
    avg_probability=("predicted_probability", "mean")
).reset_index().sort_values("decile")

decile_table["churn_rate"] = decile_table["churners"] / decile_table["customers"]
base_rate = prediction_df["true_churn"].mean()
decile_table["lift_vs_base"] = decile_table["churn_rate"] / base_rate

decile_table.to_csv(TABLE_DIR / "table_decile_lift_analysis.csv", index=False)

display(decile_table)

plt.figure(figsize=(8, 4))
sns.barplot(data=decile_table, x="decile", y="lift_vs_base", palette="viridis")
plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
plt.title("Decile Lift (Higher is Better)")
plt.xlabel("Risk decile (1 = highest risk)")
plt.ylabel("Lift vs base churn rate")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_decile_lift_analysis.png", dpi=300)
plt.show()


## 2️⃣ Explanation

### Cell purpose
This block explains *why* individual and global predictions look the way they do.

### Why it matters
Prediction without explanation is hard to trust in a dissertation and in business decisions. SHAP provides transparent feature-level attributions.


In [ ]:
# =====================================
# G4. GLOBAL SHAP DRIVER TABLE
# =====================================

shap_global_df = pd.DataFrame({
    "feature": feature_names_shap,
    "mean_abs_shap": np.abs(shap_matrix).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

shap_global_df.to_csv(TABLE_DIR / "table_shap_global_driver_ranking.csv", index=False)

print("Top global SHAP drivers:")
shap_global_df.head(15)


In [ ]:
# =====================================
# G5. LOCAL EXPLANATION TABLES FOR 3 EXAMPLE CUSTOMERS
# =====================================

example_indices = [0, min(1, X_shap.shape[0]-1), min(2, X_shap.shape[0]-1)]

for ex_id in example_indices:
    local_df = pd.DataFrame({
        "feature": feature_names_shap,
        "shap_value": shap_matrix[ex_id],
        "abs_shap": np.abs(shap_matrix[ex_id]),
        "feature_value": X_shap[ex_id],
    }).sort_values("abs_shap", ascending=False)

    local_df.head(15).to_csv(TABLE_DIR / f"table_local_shap_top15_case_{ex_id}.csv", index=False)

print("Saved local SHAP explanation tables for 3 cases.")


## 3️⃣ Insight

### Cell purpose
This block converts model outputs into structured understanding of churn behaviour.

### Why it matters
A dissertation must go beyond “model works” and show clear domain insight: which factors consistently influence churn and how patterns vary by segment.


In [ ]:
# =====================================
# G6. COMPARE GLOBAL DRIVERS ACROSS METHODS
# =====================================

# Merge Random Forest importance (if available) + permutation + SHAP for comparison.
compare_frames = []

if "rf_importances" in globals():
    temp_rf = rf_importances.copy()
    temp_rf.columns = ["feature", "rf_importance"]
    compare_frames.append(temp_rf)

if "perm_df" in globals():
    temp_perm = perm_df[["feature", "importance_mean"]].copy()
    temp_perm.columns = ["feature", "perm_importance"]
    compare_frames.append(temp_perm)

temp_shap = shap_global_df.copy()
temp_shap.columns = ["feature", "shap_importance"]
compare_frames.append(temp_shap)

# Outer merge on feature name
from functools import reduce
insight_df = reduce(lambda left, right: pd.merge(left, right, on="feature", how="outer"), compare_frames)

# Rank-normalise to compare scales if columns exist
for c in ["rf_importance", "perm_importance", "shap_importance"]:
    if c in insight_df.columns:
        insight_df[f"rank_{c}"] = insight_df[c].rank(ascending=False, method="average")

rank_cols = [c for c in insight_df.columns if c.startswith("rank_")]
if len(rank_cols) > 0:
    insight_df["avg_rank"] = insight_df[rank_cols].mean(axis=1)
    insight_df = insight_df.sort_values("avg_rank", ascending=True)

insight_df.to_csv(TABLE_DIR / "table_driver_comparison_rf_perm_shap.csv", index=False)
insight_df.head(20)


In [ ]:
# =====================================
# G7. SUBGROUP INSIGHT: CHURN RATE + MODEL PROBABILITY BY GROUP
# =====================================

group_candidates = [c for c in ["country", "gender", "active_member", "products_number", "Geography", "Gender", "IsActiveMember", "NumOfProducts"] if c in risk_df.columns]

for g in group_candidates[:6]:
    gtab = risk_df.groupby(g).agg(
        n=("true_churn", "size"),
        observed_churn=("true_churn", "mean"),
        avg_predicted_risk=("pred_proba", "mean")
    ).reset_index().sort_values("observed_churn", ascending=False)

    gtab.to_csv(TABLE_DIR / f"table_group_insight_{g}.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.barplot(data=gtab, x=g, y="observed_churn", ax=axes[0], palette="Reds")
    axes[0].set_title(f"Observed Churn by {g}")
    axes[0].tick_params(axis="x", rotation=30)

    sns.barplot(data=gtab, x=g, y="avg_predicted_risk", ax=axes[1], palette="Blues")
    axes[1].set_title(f"Predicted Risk by {g}")
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.savefig(FIG_DIR / f"fig_group_insight_{g}.png", dpi=300)
    plt.show()


## 4️⃣ Value

### Cell purpose
This block translates model outputs into business action: who to target first, what intervention strategy to use, and how to monitor outcomes.

### Why it matters
Business value is created when prediction and explanation are converted into concrete retention planning.


In [ ]:
# =====================================
# G8. ACTION PRIORITY MATRIX
# =====================================

# Build a simple intervention priority using probability + active/member information where available.
action_df = risk_df.copy()

active_like = "active_member" if "active_member" in action_df.columns else ("IsActiveMember" if "IsActiveMember" in action_df.columns else None)
products_like = "products_number" if "products_number" in action_df.columns else ("NumOfProducts" if "NumOfProducts" in action_df.columns else None)

# Simple rule-based priority labels
conditions = [
    action_df["pred_proba"] >= 0.80,
    (action_df["pred_proba"] >= 0.60) & (action_df["pred_proba"] < 0.80),
    (action_df["pred_proba"] >= 0.40) & (action_df["pred_proba"] < 0.60),
]
labels = ["Immediate", "High", "Medium"]
action_df["intervention_priority"] = np.select(conditions, labels, default="Monitor")

# Example optional action tags
if active_like is not None:
    action_df["engagement_action"] = np.where(action_df[active_like] == 0, "Engagement Campaign", "Retention Offer")
else:
    action_df["engagement_action"] = "Retention Offer"

if products_like is not None:
    action_df["product_action"] = np.where(action_df[products_like] <= 1, "Cross-sell Bundle", "Loyalty Benefits")
else:
    action_df["product_action"] = "Loyalty Benefits"

priority_table = action_df.groupby("intervention_priority").agg(
    customers=("true_churn", "size"),
    observed_churn=("true_churn", "mean"),
    avg_risk=("pred_proba", "mean")
).reset_index().sort_values("avg_risk", ascending=False)

priority_table.to_csv(TABLE_DIR / "table_action_priority_summary.csv", index=False)
action_df.to_csv(TABLE_DIR / "table_action_priority_customer_level.csv", index=False)

priority_table


In [ ]:
# =====================================
# G9. BUSINESS VALUE VISUALS
# =====================================

plt.figure(figsize=(8, 4))
sns.barplot(data=priority_table, x="intervention_priority", y="customers", palette="Purples")
plt.title("Customers by Intervention Priority")
plt.xlabel("Priority")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_customers_by_intervention_priority.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=priority_table, x="intervention_priority", y="observed_churn", palette="OrRd")
plt.title("Observed Churn Rate by Intervention Priority")
plt.xlabel("Priority")
plt.ylabel("Observed churn rate")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_churn_by_intervention_priority.png", dpi=300)
plt.show()


## How to write this section in dissertation

### Prediction
Report model ranking (F1, ROC-AUC, PR-AUC), then state how many customers are classified as high-risk and how concentrated churn is in top deciles.

### Explanation
Present SHAP global and local findings. Explain top drivers and show 1-2 customer-level cases where the model prediction is interpretable.

### Insight
Compare driver rankings across RF importance, permutation importance, and SHAP. Discuss patterns by segment (country, gender, product count, activity).

### Value
Show intervention-priority analysis and link this to potential retention strategy design (who to target first and why).


## Recommended Visual Inventory (for long dissertation)

Use this as your final checklist:

- **Prediction visuals**: class distribution, ROC, PR, confusion matrix set, probability density, decile lift
- **Explanation visuals**: SHAP bar, beeswarm, dependence plots, local waterfall cases
- **Insight visuals**: group churn/predicted risk comparisons, driver comparison tables, correlation and interaction heatmaps
- **Value visuals**: risk-segment bars, intervention-priority bars, threshold-cost curves

A strong submission typically includes **40+ visuals** and **10+ tables**, each with concise interpretation in text.


# Advanced Modelling and Explainability Extension (Methodology + Results Ready)

This extension adds advanced analytical components to strengthen dissertation quality:

- XGBoost (where available), MLP Neural Network, PCA-enhanced pipelines
- K-Fold Cross Validation for robust internal validation
- GridSearchCV for systematic hyperparameter optimisation
- Additional diagnostics for interpretation and business value

The structure below is written so outputs can be directly cited in:
- **Methodology chapter** (design, validation, tuning), and
- **Results chapter** (comparative findings and interpretation).


### Cell A1 (Methodology): Advanced model setup

This cell defines additional modelling components to improve methodological depth: a neural network classifier (MLP), PCA-enabled pipelines, and XGBoost (where the package is available). A dense preprocessing pipeline is created to ensure compatibility with PCA and MLP, both of which typically require dense feature matrices. This design supports reproducibility and fair model comparison under a shared preprocessing regime.


In [ ]:
# =====================================
# A1. ADVANCED MODEL SETUP (MLP + PCA + XGBOOST)
# =====================================

from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Dense-friendly encoder for PCA / MLP
try:
    dense_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    dense_ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

dense_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", dense_ohe)
        ]), categorical_cols),
    ]
)

advanced_extra_models = {
    "MLP Neural Network": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=400,
        random_state=RANDOM_STATE,
    )
}

if XGBOOST_AVAILABLE:
    advanced_extra_models["XGBoost Strong"] = XGBClassifier(
        n_estimators=700,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

print("Advanced extra models configured:", list(advanced_extra_models.keys()))
print("This output shows the additional advanced models that will be included in the extension.")


### Cell A2 (Methodology): K-Fold cross-validation framework

This cell applies stratified K-Fold cross-validation to advanced models using multiple performance metrics. Stratification preserves class proportions across folds, which is essential for churn data with class imbalance. Cross-validation provides a robust estimate of generalisation and reduces sensitivity to a single train-test split.


In [ ]:
# =====================================
# A2. K-FOLD CV FOR ADVANCED EXTRA MODELS
# =====================================

from sklearn.model_selection import cross_validate

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring_cv = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

cv_extra_rows = []
for name, model in advanced_extra_models.items():
    pipe = Pipeline([
        ("preprocessor", dense_preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        estimator=pipe,
        X=X_train,
        y=y_train,
        scoring=scoring_cv,
        cv=cv5,
        n_jobs=-1,
        error_score="raise"
    )

    cv_extra_rows.append({
        "Model": name,
        "CV_Accuracy": np.mean(scores["test_accuracy"]),
        "CV_Precision": np.mean(scores["test_precision"]),
        "CV_Recall": np.mean(scores["test_recall"]),
        "CV_F1": np.mean(scores["test_f1"]),
        "CV_ROC_AUC": np.mean(scores["test_roc_auc"]),
        "CV_PR_AUC": np.mean(scores["test_pr_auc"]),
    })

cv_extra_df = pd.DataFrame(cv_extra_rows).sort_values("CV_F1", ascending=False)
cv_extra_df.to_csv(TABLE_DIR / "table_cv_advanced_extra_models.csv", index=False)

display(cv_extra_df)
print("This output shows cross-validated performance of additional advanced models under a robust K-Fold framework.")


### Cell A3 (Methodology): PCA integration and variance analysis

This cell examines whether dimensionality reduction is appropriate by quantifying cumulative explained variance from PCA. In mixed-feature settings with one-hot encoding, PCA can reduce dimensional complexity and potentially improve computational efficiency. The analysis is included as a methodological justification for PCA-based model variants.


In [ ]:
# =====================================
# A3. PCA EXPLAINED VARIANCE ANALYSIS
# =====================================

X_train_dense = dense_preprocessor.fit_transform(X_train)

# limit components for efficiency; if very wide, cap at 60
max_components = min(60, X_train_dense.shape[1])
pca_probe = PCA(n_components=max_components, random_state=RANDOM_STATE)
pca_probe.fit(X_train_dense)

cum_var = np.cumsum(pca_probe.explained_variance_ratio_)

pca_var_df = pd.DataFrame({
    "n_components": np.arange(1, max_components + 1),
    "cumulative_explained_variance": cum_var
})
pca_var_df.to_csv(TABLE_DIR / "table_pca_cumulative_variance.csv", index=False)

# choose component count for >=95% variance
n95 = int(np.argmax(cum_var >= 0.95) + 1) if np.any(cum_var >= 0.95) else max_components

plt.figure(figsize=(8, 5))
plt.plot(pca_var_df["n_components"], pca_var_df["cumulative_explained_variance"], marker="o")
plt.axhline(0.95, linestyle="--", color="red", label="95% variance")
plt.axvline(n95, linestyle="--", color="black", label=f"n={n95}")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Cumulative Explained Variance")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_pca_cumulative_explained_variance.png", dpi=300)
plt.show()

print(f"Selected PCA component count (>=95% variance): {n95}")
print("This plot shows how much information is retained as the number of principal components increases.")


### Cell A4 (Methodology): GridSearchCV hyperparameter tuning

This cell applies GridSearchCV to tune advanced models (MLP and optional XGBoost) under a consistent CV protocol. Grid search provides a transparent and systematic optimisation approach suitable for academic reporting. The objective metric is F1-score, reflecting balanced concern for precision and recall in churn prediction.


In [ ]:
# =====================================
# A4. GRIDSEARCHCV FOR MLP / XGBOOST
# =====================================

grid_results = []
grid_best_pipelines = {}

# MLP grid
mlp_pipe = Pipeline([
    ("preprocessor", dense_preprocessor),
    ("pca", PCA(n_components=n95, random_state=RANDOM_STATE)),
    ("model", MLPClassifier(max_iter=500, random_state=RANDOM_STATE))
])

mlp_grid = {
    "model__hidden_layer_sizes": [(64,), (64, 32), (128, 64)],
    "model__alpha": [1e-4, 1e-3],
    "model__learning_rate_init": [1e-3, 5e-4],
}

mlp_search = GridSearchCV(
    estimator=mlp_pipe,
    param_grid=mlp_grid,
    scoring="f1",
    cv=cv5,
    n_jobs=-1,
    verbose=0
)
mlp_search.fit(X_train, y_train)

grid_best_pipelines["MLP+PCA (Grid)"] = mlp_search.best_estimator_
grid_results.append({
    "Model": "MLP+PCA (Grid)",
    "BestCV_F1": mlp_search.best_score_,
    "BestParams": str(mlp_search.best_params_)
})

# XGBoost grid (optional)
if XGBOOST_AVAILABLE:
    xgb_pipe = Pipeline([
        ("preprocessor", dense_preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ])

    xgb_grid = {
        "model__n_estimators": [300, 500],
        "model__learning_rate": [0.03, 0.05],
        "model__max_depth": [3, 4, 6],
        "model__subsample": [0.8, 0.9],
        "model__colsample_bytree": [0.8, 0.9],
    }

    xgb_search = GridSearchCV(
        estimator=xgb_pipe,
        param_grid=xgb_grid,
        scoring="f1",
        cv=cv5,
        n_jobs=-1,
        verbose=0
    )
    xgb_search.fit(X_train, y_train)

    grid_best_pipelines["XGBoost (Grid)"] = xgb_search.best_estimator_
    grid_results.append({
        "Model": "XGBoost (Grid)",
        "BestCV_F1": xgb_search.best_score_,
        "BestParams": str(xgb_search.best_params_)
    })

grid_df = pd.DataFrame(grid_results).sort_values("BestCV_F1", ascending=False)
grid_df.to_csv(TABLE_DIR / "table_gridsearch_summary_advanced.csv", index=False)

display(grid_df)
print("This output shows the best cross-validated hyperparameter settings and performance for tuned advanced models.")


### Cell A5 (Results): Evaluate tuned advanced models on hold-out test set

This cell evaluates the tuned advanced pipelines on unseen test data and compares them using a multi-metric framework. Reporting both discrimination and error-balance metrics supports robust academic interpretation of practical model quality.


In [ ]:
# =====================================
# A5. TEST EVALUATION FOR GRID-TUNED MODELS
# =====================================

grid_eval_rows = []
grid_roc = {}
grid_pr = {}

for name, pipe in grid_best_pipelines.items():
    m, cm, fpr, tpr, prec, rec, y_proba = evaluate_model_extended(name, pipe, X_test, y_test)
    grid_eval_rows.append(m)
    grid_roc[name] = (fpr, tpr, m["ROC_AUC"])
    grid_pr[name] = (rec, prec, m["PR_AUC"])

grid_eval_df = pd.DataFrame(grid_eval_rows).sort_values(["F1", "ROC_AUC"], ascending=[False, False])
grid_eval_df.to_csv(TABLE_DIR / "table_grid_models_test_metrics.csv", index=False)

display(grid_eval_df)
print("This output shows the hold-out test performance of GridSearch-optimised advanced models.")


### Cell A6 (Results): Baseline vs advanced tuned comparison

This cell integrates the best results from previous sections to provide a single consolidated comparison table. This is intended to be the principal model-comparison table for dissertation reporting.


In [ ]:
# =====================================
# A6. CONSOLIDATED MODEL COMPARISON TABLE
# =====================================

# Reuse all_eval_df from earlier extension and append grid-evaluated models
comparison_frames = [all_eval_df.copy()]
if len(grid_eval_rows) > 0:
    comparison_frames.append(grid_eval_df.copy())

final_comp_df = pd.concat(comparison_frames, ignore_index=True)
final_comp_df = final_comp_df.sort_values(["F1", "ROC_AUC"], ascending=[False, False]).drop_duplicates(subset=["Model"])
final_comp_df.to_csv(TABLE_DIR / "table_final_model_comparison_including_grid.csv", index=False)

display(final_comp_df.head(20))
print("This output shows a unified performance ranking across baseline, advanced, tuned, and GridSearch-optimised models.")


### Cell A7 (Results): Comparative visual diagnostics for tuned models

This cell produces ROC and precision-recall visual comparisons for the tuned advanced models. These plots supplement numeric tables by illustrating discrimination quality and minority-class retrieval behaviour across thresholds.


In [ ]:
# =====================================
# A7. ROC + PR PLOTS FOR GRID-TUNED MODELS
# =====================================

if len(grid_roc) > 0:
    plt.figure(figsize=(8, 6))
    for m, (fpr, tpr, auc) in grid_roc.items():
        plt.plot(fpr, tpr, label=f"{m} (AUC={auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves: Grid-Tuned Advanced Models")
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_roc_grid_tuned_models.png", dpi=300)
    plt.show()
    print("This plot shows discrimination performance of Grid-tuned advanced models across thresholds.")

    plt.figure(figsize=(8, 6))
    for m, (rec, prec, ap) in grid_pr.items():
        plt.plot(rec, prec, label=f"{m} (AP={ap:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curves: Grid-Tuned Advanced Models")
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_pr_grid_tuned_models.png", dpi=300)
    plt.show()
    print("This plot shows precision-recall trade-offs for Grid-tuned models, which is especially relevant under class imbalance.")
else:
    print("No grid-tuned models were available for ROC/PR plotting.")
    print("This output shows that optional GridSearch models were not produced in the current run configuration.")


### Cell A8 (Results + Discussion): PCA model fit justification

This cell compares tuned MLP with PCA against non-PCA references using the consolidated table. The aim is to evaluate whether dimensionality reduction contributes to measurable performance gain or computational simplification. This supports critical methodological discussion rather than assuming PCA is universally beneficial.


In [ ]:
# =====================================
# A8. PCA FIT CHECK IN FINAL RANKING
# =====================================

pca_related = final_comp_df[final_comp_df["Model"].str.contains("PCA|MLP", case=False, regex=True)].copy()
if len(pca_related) > 0:
    pca_related.to_csv(TABLE_DIR / "table_pca_related_model_performance.csv", index=False)
    display(pca_related)
    print("This output shows how PCA-related and neural-network variants perform relative to other candidate models.")
else:
    print("No PCA-related models were found in the final comparison table.")
    print("This output shows that PCA-enabled models were not present in the evaluated set for this run.")


### Cell A9 (Results): Update best-model selection after advanced extension

This cell re-selects the best-performing model after inclusion of GridSearch-tuned advanced variants. It ensures that downstream explanation and business-value analysis reflect the strongest available model evidence.


In [ ]:
# =====================================
# A9. UPDATE BEST MODEL AFTER ADVANCED EXTENSION
# =====================================

best_model_name_final = final_comp_df.iloc[0]["Model"]

# Resolve pipeline object from available dictionaries
best_model_pipe_final = None
for d in [all_candidate_models, grid_best_pipelines]:
    if best_model_name_final in d:
        best_model_pipe_final = d[best_model_name_final]
        break

if best_model_pipe_final is None:
    raise ValueError("Could not resolve final best model pipeline.")

print("Final selected best model:", best_model_name_final)
print("This output shows the final model selected after incorporating advanced and tuned model candidates.")


### Cell A10 (Results): Final interpretive summary table for dissertation

This cell creates a concise, chapter-ready summary table containing the top-performing model and key evaluation metrics. The table is intended for direct use in the results chapter and supports concise reporting of principal findings.


In [ ]:
# =====================================
# A10. DISSERTATION-READY TOP MODEL SUMMARY
# =====================================

top_summary = final_comp_df.head(5).copy()
top_summary.to_csv(TABLE_DIR / "table_top5_models_dissertation_summary.csv", index=False)

display(top_summary)
print("This output shows the final top-performing models and their key metrics for dissertation reporting.")


## Dissertation Use Notes for This Advanced Extension

- **Methodology chapter:** cite Cells A1-A4 for model architecture, cross-validation design, PCA rationale, and hyperparameter tuning protocol.
- **Results chapter:** cite Cells A5-A10 for comparative outcomes, plots, and final model selection evidence.
- For every inserted table or figure, use the generated CSV/PNG filenames as traceable artefacts and add corresponding dissertation figure/table numbers.


# Final Ordered Workflow for Dissertation Submission

This section provides a clean and correct order for final reporting:

1. Compare **all available models** under a unified evaluation framework.
2. Select the **final best model** using objective ranking criteria.
3. Apply explainability methods **after** final model selection.
4. Produce final business-facing risk outputs from the selected model.

> For dissertation writing, treat earlier modelling blocks as development/exploration and use this final ordered section for Methodology and Results reporting.


### Cell F1 (Methodology): Unified model pool assembly

This cell consolidates every trained pipeline from baseline, advanced, tuned, and GridSearch sections into a single candidate pool. This ensures that model comparison is comprehensive and that final model selection is performed only after evaluating all feasible alternatives.


In [ ]:
# =====================================
# F1. BUILD UNIFIED MODEL CANDIDATE POOL
# =====================================

final_model_pool = {}

# Safely include model dictionaries if they exist
for var_name in ["trained_pipelines", "advanced_pipelines", "tuned_pipelines", "grid_best_pipelines"]:
    if var_name in globals() and isinstance(globals()[var_name], dict):
        final_model_pool.update(globals()[var_name])

if len(final_model_pool) == 0:
    raise ValueError("No trained model pipelines found. Please run prior modelling cells first.")

print(f"Total models in final candidate pool: {len(final_model_pool)}")
for k in final_model_pool.keys():
    print(" -", k)
print("This output shows that all model families have been consolidated before final comparison.")


### Cell F2 (Results): Final all-model comparison table

This cell evaluates all models in the unified candidate pool using a consistent metric set. The resulting table is the definitive comparison artefact for results reporting and should be used as the basis for final model selection.


In [ ]:
# =====================================
# F2. FINAL ALL-MODEL EVALUATION TABLE
# =====================================

final_rows = []
final_roc_data = {}
final_pr_data = {}
final_cm_data = {}
final_proba_data = {}

for model_name, model_pipe in final_model_pool.items():
    m, cm, fpr, tpr, prec, rec, y_proba = evaluate_model_extended(model_name, model_pipe, X_test, y_test)
    final_rows.append(m)
    final_roc_data[model_name] = (fpr, tpr, m["ROC_AUC"])
    final_pr_data[model_name] = (rec, prec, m["PR_AUC"])
    final_cm_data[model_name] = cm
    final_proba_data[model_name] = y_proba

final_all_models_df = pd.DataFrame(final_rows)
final_all_models_df = final_all_models_df.sort_values(["F1", "ROC_AUC", "PR_AUC"], ascending=[False, False, False])
final_all_models_df.to_csv(TABLE_DIR / "table_FINAL_all_models_comparison.csv", index=False)

display(final_all_models_df)
print("This output shows the final ranked comparison of all models under the same evaluation criteria.")


### Cell F3 (Results): Final model selection

This cell applies an explicit ranking rule to select the final best model after all comparisons are complete. The selected model is then used for all subsequent explainability and business-value analyses.


In [ ]:
# =====================================
# F3. SELECT FINAL BEST MODEL
# =====================================

FINAL_MODEL_NAME = final_all_models_df.iloc[0]["Model"]
FINAL_MODEL_PIPE = final_model_pool[FINAL_MODEL_NAME]
FINAL_MODEL_PROBA = final_proba_data[FINAL_MODEL_NAME]

print("Final selected model:", FINAL_MODEL_NAME)
print("This output shows the model selected only after evaluating the full candidate set.")


### Cell F4 (Results): Final explainability on selected model

This cell computes explainability outputs using the **final selected model only**. This preserves interpretive consistency and ensures that all XAI conclusions correspond directly to the model chosen in the final comparison stage.


In [ ]:
# =====================================
# F4. FINAL MODEL EXPLAINABILITY (SHAP + PERMUTATION)
# =====================================

# 1) Permutation importance on final model
final_perm = permutation_importance(
    estimator=FINAL_MODEL_PIPE,
    X=X_test,
    y=y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scoring="f1"
)

final_perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": final_perm.importances_mean,
    "importance_std": final_perm.importances_std,
}).sort_values("importance_mean", ascending=False)
final_perm_df.to_csv(TABLE_DIR / "table_FINAL_permutation_importance.csv", index=False)

display(final_perm_df.head(20))
print("This output shows the most influential final-model features under model-agnostic permutation analysis.")

# 2) SHAP if final model is tree-compatible, else fallback to best available tree model
final_model_name_lower = FINAL_MODEL_NAME.lower()
final_tree_like = any(k in final_model_name_lower for k in ["forest", "boost", "xgboost", "tree", "hist"])

if final_tree_like:
    shap_pipe_final = FINAL_MODEL_PIPE
    shap_used_name = FINAL_MODEL_NAME
else:
    # fallback preference for tree models in pool
    fallback_name = None
    for candidate in final_model_pool.keys():
        c = candidate.lower()
        if any(k in c for k in ["forest", "boost", "xgboost", "tree", "hist"]):
            fallback_name = candidate
            break

    if fallback_name is None:
        shap_pipe_final = FINAL_MODEL_PIPE
        shap_used_name = FINAL_MODEL_NAME
    else:
        shap_pipe_final = final_model_pool[fallback_name]
        shap_used_name = fallback_name

print("SHAP model used for final explanation:", shap_used_name)

shap_pre_final = shap_pipe_final.named_steps["preprocessor"]
shap_model_final = shap_pipe_final.named_steps["model"]
shap_features_final = shap_pre_final.get_feature_names_out()
X_test_trans_final = shap_pre_final.transform(X_test)
if hasattr(X_test_trans_final, "toarray"):
    X_test_trans_final = X_test_trans_final.toarray()

sample_n_final = min(400, X_test_trans_final.shape[0])
sample_idx_final = np.random.RandomState(RANDOM_STATE).choice(X_test_trans_final.shape[0], sample_n_final, replace=False)
X_shap_final = X_test_trans_final[sample_idx_final]

explainer_final = shap.TreeExplainer(shap_model_final)
shap_values_final = explainer_final.shap_values(X_shap_final)

if isinstance(shap_values_final, list) and len(shap_values_final) == 2:
    shap_matrix_final = shap_values_final[1]
elif isinstance(shap_values_final, np.ndarray) and shap_values_final.ndim == 3:
    shap_matrix_final = shap_values_final[:, :, 1]
else:
    shap_matrix_final = shap_values_final

final_shap_df = pd.DataFrame({
    "feature": shap_features_final,
    "mean_abs_shap": np.abs(shap_matrix_final).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

final_shap_df.to_csv(TABLE_DIR / "table_FINAL_shap_global_ranking.csv", index=False)

plt.figure()
shap.summary_plot(
    shap_matrix_final,
    X_shap_final,
    feature_names=shap_features_final,
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_FINAL_shap_bar.png", dpi=300, bbox_inches="tight")
plt.show()
print("This plot shows the global SHAP feature ranking for the final selected model workflow.")

plt.figure()
shap.summary_plot(
    shap_matrix_final,
    X_shap_final,
    feature_names=shap_features_final,
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_FINAL_shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()
print("This plot shows the distribution and direction of SHAP feature effects across customers.")


### Cell F5 (Results + Value): Final risk targeting outputs from selected model

This cell produces the final customer-level risk ranking and decile lift from the selected model, ensuring that business-facing outputs are tied to the same model used in final academic interpretation.


In [ ]:
# =====================================
# F5. FINAL RISK TARGETING TABLES (SELECTED MODEL)
# =====================================

final_pred_df = X_test.copy()
final_pred_df["true_churn"] = y_test.values
final_pred_df["predicted_probability"] = FINAL_MODEL_PROBA
final_pred_df["predicted_label_0_5"] = (final_pred_df["predicted_probability"] >= 0.5).astype(int)

# Top 100 likely churners
final_top100 = final_pred_df.sort_values("predicted_probability", ascending=False).head(100)
final_top100.to_csv(TABLE_DIR / "table_FINAL_top100_likely_churners.csv", index=False)

# Decile lift
final_decile = final_pred_df.copy()
final_decile["decile"] = pd.qcut(final_decile["predicted_probability"], q=10, labels=False, duplicates="drop")
final_decile["decile"] = final_decile["decile"].max() - final_decile["decile"] + 1

final_decile_table = final_decile.groupby("decile").agg(
    customers=("true_churn", "size"),
    churners=("true_churn", "sum"),
    avg_probability=("predicted_probability", "mean")
).reset_index().sort_values("decile")

final_decile_table["churn_rate"] = final_decile_table["churners"] / final_decile_table["customers"]
final_base = final_pred_df["true_churn"].mean()
final_decile_table["lift_vs_base"] = final_decile_table["churn_rate"] / final_base
final_decile_table.to_csv(TABLE_DIR / "table_FINAL_decile_lift.csv", index=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=final_decile_table, x="decile", y="lift_vs_base", palette="viridis")
plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
plt.title(f"Final Model Decile Lift ({FINAL_MODEL_NAME})")
plt.xlabel("Risk decile (1 = highest risk)")
plt.ylabel("Lift vs base churn rate")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_FINAL_decile_lift.png", dpi=300)
plt.show()

display(final_decile_table)
print("This output shows the final model’s concentration of churn risk across deciles for practical targeting.")
print("This plot shows how much better each risk decile performs compared with the base churn rate.")


## Final dissertation instruction for ordering

For submission, write results using this final ordered section (F1-F5):

1. **F2 table** for all-model comparison
2. **F3** for final model selection
3. **F4** for explainability findings on selected model
4. **F5** for business-targeting outputs tied to selected model

This ensures methodological correctness: model comparison precedes model selection, and explainability is conducted on the selected final model.
